## Using `xarray.map_blocks` with spatial data

`map_blocks` applies a **Python function once per chunk (block)** of a Dask-backed DataArray or Dataset.
Each call receives an in-memory xarray object for that block; the returned blocks are stitched back into one lazy result.

### When to use it

Use `map_blocks` when:

1. You need **custom Python / NumPy / SciPy** logic that is not a clean chain of xarray ops.
2. The function naturally works on a **spatial tile / chunk** (same shape in, or a known shape out).
3. You still want **Dask parallelism** over chunks (local cluster or Gateway).
4. You are wrapping code that expects a small NumPy array or an xarray *subset*, not a global Dask graph.

### When *not* to use it

- The work is already expressible as xarray / Dask Array ops (`.where`, `/`, `.mean`, …) — use those instead.
- You need a **global** view of the whole array at once.
- The algorithm needs **neighbourhoods that cross chunk boundaries** (filters, focal stats) unless you rechunk / overlap carefully — naive `map_blocks` only sees one chunk.
- You want arbitrary side-effect jobs (write files, call APIs) with dynamic control flow — prefer **Futures** / **Delayed**.

### Mental model

| Tool | Best for |
|---|---|
| Native xarray / Dask Array | Array math already supported by the libraries |
| `map_blocks` | Custom per-chunk functions on chunked xarray |
| `apply_ufunc` | NumPy ufuncs / vectorised core funcs over dims |
| Futures / Delayed | General-purpose Python pipelines, not chunk graphs |

In [1]:
import datacube
from dask.distributed import Client, LocalCluster
import matplotlib.pyplot as plt
from numba import njit
import numpy as np
import xarray as xr

In [2]:
# Initialise datacube

dc = datacube.Datacube()

In [3]:
# (Central NSW)
x_min, x_max = 1200000, 1300000  # 100km wide
y_min, y_max = -3600000, -3700000  # 100km high
date_range = ("2024-01-01", "2024-02-28")

### Load chunked spatial data (lazy)

Chunks define the blocks that `map_blocks` will iterate over. The load below uses `time: 1` and `y`/`x`: 500 — good for streaming IO, but **not** yet ideal for a per-tile temporal mean (see rechunk step later).

In [4]:
product = "ga_s2bm_ard_3"  # Sentinel-2 B
measurements = ["nbart_red", "nbart_blue", "oa_s2cloudless_mask"]
output_crs = "EPSG:3577"
resolution = [-30, 30]

dask_chunks = {
    "time": 1,
    "y": 500,
    "x": 500,
}

ds = dc.load(
    product=product,
    measurements=measurements,
    crs="EPSG:3577",
    x=(x_min, x_max),
    y=(y_min, y_max),
    time=date_range,
    output_crs=output_crs,
    resolution=resolution,
    dask_chunks=dask_chunks,
    dataset_predicate=lambda ds: ds.metadata.dataset_maturity == "final",
    skip_broken_datasets=True,
)
ds

<xarray.Dataset> Size: 2GB
Dimensions:              (time: 30, y: 3334, x: 3334)
Coordinates:
  * time                 (time) datetime64[ns] 240B 2024-01-10T00:25:45.71496...
  * y                    (y) float64 27kB -3.6e+06 -3.6e+06 ... -3.7e+06
  * x                    (x) float64 27kB 1.2e+06 1.2e+06 ... 1.3e+06 1.3e+06
    spatial_ref          int32 4B 3577
Data variables:
    nbart_red            (time, y, x) int16 667MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    nbart_blue           (time, y, x) int16 667MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    oa_s2cloudless_mask  (time, y, x) uint8 333MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [5]:
# chunks is a tuple of (time_chunks, y_chunks, x_chunks)
# Example:
#   time: 30 chunks of size 1  → one date per block
#   y/x:  6×500 + one remainder (e.g. 334) → AOI not evenly divisible by 500
#
# With time chunked as 1, map_blocks would NOT see the full time series in one call.
ds["nbart_red"].chunks

((1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1),
 (500, 500, 500, 500, 500, 500, 334),
 (500, 500, 500, 500, 500, 500, 334))

### Start a local Dask cluster

In [6]:
cluster = LocalCluster()
client = Client(cluster)
print(client.dashboard_link)
client

/user/csiro-csiro-aad_bri246@csiro.au/proxy/8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/csiro-csiro-aad_bri246@csiro.au/proxy/8787/status,
Dashboard: /user/csiro-csiro-aad_bri246@csiro.au/proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 8.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35473,Workers: 0
Dashboard: /user/csiro-csiro-aad_bri246@csiro.au/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39695,Total threads: 1
Dashboard: /user/csiro-csiro-aad_bri246@csiro.au/proxy/43699/status,Memory: 2.00 GiB
Nanny: tcp://127.0.0.1:39073,


### Example 1 — Prefer native xarray when you can

Cloud-masked red/blue mean ratio is plain array math. **Do this with xarray**, not `map_blocks`.

In [7]:
no_clouds = ds.where(ds["oa_s2cloudless_mask"] == 1)
ratio = no_clouds["nbart_red"] / no_clouds["nbart_blue"]
mean_ratio_native = ratio.mean(dim="time", skipna=True)
mean_ratio_native

<xarray.DataArray (y: 3334, x: 3334)> Size: 44MB
dask.array<mean_agg-aggregate, shape=(3334, 3334), dtype=float32, chunksize=(500, 500), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 27kB -3.6e+06 -3.6e+06 ... -3.7e+06 -3.7e+06
  * x            (x) float64 27kB 1.2e+06 1.2e+06 1.2e+06 ... 1.3e+06 1.3e+06
    spatial_ref  int32 4B 3577

### Rechunk so each block has the data your function needs

Your chunk layout was roughly:

- **time:** many chunks of size `1` (one date each)
- **y / x:** mostly `500`, with a smaller edge chunk (e.g. `334`)

If the function does a **temporal mean**, each block must contain **all times** for that spatial window. Rechunk `time` to `-1` (one chunk along time) while keeping spatial tiles:

In [8]:
ds_tiles = ds.chunk({"time": -1, "y": 500, "x": 500})
ds_tiles["nbart_red"].chunks  # time should now be a single chunk equal to the full length

((30,),
 (500, 500, 500, 500, 500, 500, 334),
 (500, 500, 500, 500, 500, 500, 334))

### Example 2 — Same idea via `map_blocks` (for illustration)

This recreates the ratio+mean **inside a per-block function**. Functionally fine, but more verbose than native ops.

Because the output drops `time`, we must pass a **`template`** that describes the result shape/dtype/coords.

Use `ds_tiles` (all times in each spatial chunk), not the original `time: 1` chunking.

In [9]:
def mean_ratio_block(block: xr.Dataset) -> xr.DataArray:
    """
    Runs on one spatial tile at a time, with the full time series present
    (eager NumPy-backed xarray inside the worker).
    """
    clear = block.where(block["oa_s2cloudless_mask"] == 1)
    ratio = clear["nbart_red"] / clear["nbart_blue"]
    return ratio.mean(dim="time", skipna=True)


# Template: same spatial grid as a time-reduced DataArray
template = xr.full_like(ds_tiles["nbart_red"].isel(time=0, drop=True), fill_value=np.nan, dtype="float64")

mean_ratio_mapped = xr.map_blocks(
    mean_ratio_block,  # function to apply to each block
    ds_tiles,  # input data
    template=template,  # tells system what the output will look like (it's array dims)
)
mean_ratio_mapped

<xarray.DataArray 'nbart_red' (y: 3334, x: 3334)> Size: 89MB
dask.array<<this-array>-mean_ratio_block, shape=(3334, 3334), dtype=float64, chunksize=(500, 500), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 27kB -3.6e+06 -3.6e+06 ... -3.7e+06 -3.7e+06
  * x            (x) float64 27kB 1.2e+06 1.2e+06 1.2e+06 ... 1.3e+06 1.3e+06
    spatial_ref  int32 4B dask.array<chunksize=(), meta=np.ndarray>
Attributes:
    units:         1
    nodata:        -999
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

### Example 3 — Better reason: a Numba kernel via `map_blocks`

Here the chunk is only a **batch** for parallelism/memory — not a statistical unit.

We run a **Numba** kernel that, for each pixel, walks the time axis with custom branching:

- keep clear pixels only (`oa_s2cloudless_mask == 1`)
- form red/blue ratio (skip invalid / zero blue)
- require at least `min_clear` observations, else NaN

That kind of nested pixel loop is awkward in pure xarray and slow in plain Python; Numba compiles it, and `map_blocks` runs one spatial tile per task.

**Gotcha:** the returned block must match the template's coordinates, including non-dimension ones such as `spatial_ref` (the CRS). Building a `DataArray` from scratch with only `y`/`x` raises:

```
ValueError: Result from applying user function does not contain coordinate variables {'spatial_ref'}.
```

The fix below reuses a time-reduced slice of the block (`isel(time=0, drop=True)`) for `dims` and `coords`, so all coords come along automatically.

In [10]:
@njit
def clear_mean_ratio_kernel(red, blue, mask, min_clear):
    """
    Per-pixel clear-sky mean of red/blue.

    red, blue, mask: arrays shaped (time, y, x)
    returns: (y, x)
    """
    n_time, n_y, n_x = red.shape
    out = np.empty((n_y, n_x), dtype=np.float64)

    for i in range(n_y):
        for j in range(n_x):
            total = 0.0
            count = 0
            for t in range(n_time):
                if mask[t, i, j] != 1:
                    continue
                r = red[t, i, j]
                b = blue[t, i, j]
                if np.isfinite(r) and np.isfinite(b) and b != 0:
                    total += r / b
                    count += 1
            if count >= min_clear:
                out[i, j] = total / count
            else:
                out[i, j] = np.nan
    return out


def clear_mean_ratio_block(block: xr.Dataset, min_clear: int = 3) -> xr.DataArray:
    """map_blocks wrapper: pull NumPy arrays, call Numba, wrap result as DataArray."""
    red = block["nbart_red"].values.astype(np.float64)
    blue = block["nbart_blue"].values.astype(np.float64)
    mask = block["oa_s2cloudless_mask"].values

    out = clear_mean_ratio_kernel(red, blue, mask, min_clear)

    # Reuse a time-reduced slice of the block so non-dimension coords
    # (e.g. spatial_ref) are carried through; map_blocks compares the
    # returned object against the template and errors if any are missing.
    like = block["nbart_red"].isel(time=0, drop=True)

    return xr.DataArray(
        out,
        dims=like.dims,
        coords=like.coords,
        name="clear_mean_ratio",
    )


template = xr.full_like(
    ds_tiles["nbart_red"].isel(time=0, drop=True),
    fill_value=np.nan,
    dtype="float64",
)
template.name = "clear_mean_ratio"

clear_mean_ratio = xr.map_blocks(
    clear_mean_ratio_block,
    ds_tiles,
    kwargs={"min_clear": 3},
    template=template,
)
clear_mean_ratio

<xarray.DataArray 'clear_mean_ratio' (y: 3334, x: 3334)> Size: 89MB
dask.array<<this-array>-clear_mean_ratio_block, shape=(3334, 3334), dtype=float64, chunksize=(500, 500), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 27kB -3.6e+06 -3.6e+06 ... -3.7e+06 -3.7e+06
  * x            (x) float64 27kB 1.2e+06 1.2e+06 1.2e+06 ... 1.3e+06 1.3e+06
    spatial_ref  int32 4B dask.array<chunksize=(), meta=np.ndarray>
Attributes:
    units:         1
    nodata:        -999
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [ ]:
%%time
result = clear_mean_ratio.compute()
result

In [ ]:
band = result

band.plot.imshow(cmap="viridis")
plt.title("Clear-sky mean red/blue (Numba + map_blocks)")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### Chunk boundaries (important for spatial algorithms)

`map_blocks` sees **one chunk at a time**. A 3×3 focal filter, contour tracer, or anything that needs neighbouring pixels **across chunk edges** will be wrong at boundaries unless you:

- rechunk so each AOI of interest is one chunk, or
- use overlapping chunks / ghost cells (more advanced), or
- run the neighbourhood op with a tool that handles overlaps explicitly.

Per-pixel kernels (like the Numba example above) are a good fit: each pixel only uses its own time series, so spatial chunk edges are fine.

### Clean up

In [ ]:
client.close()
cluster.close()

### Takeaways

1. **`map_blocks`** = run a Python function on each chunk of a Dask-backed xarray object, in parallel.
2. **Chunking must match the function:** if you reduce over `time`, each block needs all times (`chunk({"time": -1, ...})`).
3. Strong use case: wrap **Numba / NumPy / SciPy** kernels that expect arrays, while Dask parallelises over spatial tiles.
4. Prefer **native xarray** for mask / arithmetic / reductions when that is enough.
5. Provide a **`template`** when the output dims/shape/dtype differ from the input, and make each returned block match it — including non-dimension coords like `spatial_ref`.
6. Mind **chunk boundaries** for any algorithm that needs neighbouring pixels across tiles.
7. For non-array general Python workflows, use **Futures** or **Delayed** instead.